# Eye Tracking Fundamentals
### From raw iris offset → gaze-in-head → calibrated screen gaze

This notebook builds a proper eye tracking pipeline step by step.

**The problem with naive iris tracking:**
If you just measure where the iris is in the camera frame, any head movement — rotation, translation, leaning — changes the iris position even if your eyes aren't moving. This makes it useless as a gaze signal.

**The solution — gaze-in-head:**
Measure iris position in the eye's own local coordinate system. Build a coordinate frame using the eye corners as axes. Now head movement doesn't matter — the measurement is always relative to the eye itself.

**Pipeline:**
```
Raw iris pixel position
        ↓
Local eye coordinate system
        ↓
Gaze-in-head (stable, head-invariant)
        ↓
Smoothing
        ↓
Calibration mapping (gaze → screen)
        ↓
Screen gaze point
```

Run cells in order. Make sure `face_landmarker.task` and `hand_landmarker.task` are in the same folder.

## Cell 1 — Imports & Setup

We need:
- `cv2` — frame capture and drawing
- `mediapipe` — face and hand landmarks
- `numpy` — vector math for coordinate systems
- `time` — MediaPipe VIDEO mode requires timestamps

In [3]:
import cv2
import numpy as np
import mediapipe as mp
import time
import os

BASE_DIR = os.getcwd()
print(f'Working directory: {BASE_DIR}')

Working directory: c:\Coding\Python\internship\Pytorch\M6 Video & Motion Recognition\Mini-Project-3


## Cell 2 — MediaPipe Landmarker Setup

We run two landmarkers simultaneously:
- **FaceLandmarker** — 478 landmarks including iris centers (468–477)
- **HandLandmarker** — 21 landmarks per hand for pinch detection

Both use `VIDEO` mode which requires a monotonically increasing timestamp per frame.
This is more efficient than `LIVE_STREAM` mode for a notebook.

In [4]:
BaseOptions           = mp.tasks.BaseOptions
FaceLandmarker        = mp.tasks.vision.FaceLandmarker
FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions
HandLandmarker        = mp.tasks.vision.HandLandmarker
HandLandmarkerOptions = mp.tasks.vision.HandLandmarkerOptions
VisionRunningMode     = mp.tasks.vision.RunningMode

face_options = FaceLandmarkerOptions(
    base_options = BaseOptions(model_asset_path=os.path.join(BASE_DIR, '../face_landmarker.task')),
    running_mode = VisionRunningMode.VIDEO,
    num_faces    = 1
)

hand_options = HandLandmarkerOptions(
    base_options = BaseOptions(model_asset_path=os.path.join(BASE_DIR, '../hand_landmarker.task')),
    running_mode = VisionRunningMode.VIDEO,
    num_hands    = 1,
    min_hand_detection_confidence = 0.7,
    min_hand_presence_confidence  = 0.7,
    min_tracking_confidence       = 0.7
)

face_landmarker = FaceLandmarker.create_from_options(face_options)
hand_landmarker = HandLandmarker.create_from_options(hand_options)

print('Face and hand landmarkers ready!')

Face and hand landmarkers ready!


## Cell 3 — Landmark Indices & Constants

MediaPipe gives us 478 face landmarks. We only need a small subset:

```
LEFT EYE:
  468 = Left iris center
  33  = Left eye inner corner  (closest to nose)
  133 = Left eye outer corner  (closest to ear)
  159 = Left eye top
  145 = Left eye bottom

RIGHT EYE:
  473 = Right iris center
  362 = Right eye inner corner
  263 = Right eye outer corner
  386 = Right eye top
  374 = Right eye bottom
```

For hand pinch we use:
```
  4  = Thumb tip
  8  = Index tip
  12 = Middle tip   ← fallback if index goes off screen
```

In [5]:
# Left eye
L_IRIS   = 468
L_INNER  = 33
L_OUTER  = 133
L_TOP    = 159
L_BOTTOM = 145

# Right eye
R_IRIS   = 473
R_INNER  = 362
R_OUTER  = 263
R_TOP    = 386
R_BOTTOM = 374

# Hand pinch
THUMB_TIP  = 4
INDEX_TIP  = 8
MIDDLE_TIP = 12

# Pinch threshold — normalized distance between thumb and finger
PINCH_THRESHOLD = 0.05

# Smoothing window
SMOOTH_WINDOW = 10

# Colors
WHITE  = (255, 255, 255)
GREEN  = (0, 255, 0)
YELLOW = (0, 255, 255)
RED    = (0, 0, 255)
CYAN   = (255, 255, 0)
ORANGE = (0, 165, 255)

print('Constants set!')

Constants set!


## Cell 4 — Eye Local Coordinate System

### The Problem
MediaPipe gives us iris position as normalized screen coordinates `(x, y)` — values between 0 and 1 relative to the full camera frame. This means:
- If you move your head left, iris X decreases — even if your eyes didn't move
- If you rotate your head, the eye corners tilt — the iris appears to shift

### The Solution — Local Coordinate Frame
Instead of measuring iris position in camera space, we build a coordinate frame **attached to the eye itself** using the eye corners as basis vectors.

```
    inner_corner ────────────── outer_corner
         ↑                           ↑
     landmark[L_INNER]         landmark[L_OUTER]
```

**Step 1 — Eye center:**
```
eye_center = (inner_corner + outer_corner) / 2
```

**Step 2 — X axis (horizontal):**
```
x_axis = outer_corner - inner_corner   (points from inner to outer)
x_axis = x_axis / |x_axis|            (normalize to unit vector)
```

**Step 3 — Y axis (vertical):**
```
y_axis = top - bottom                  (points upward)
y_axis = y_axis / |y_axis|            (normalize to unit vector)
```

**Step 4 — Project iris onto axes:**
```
iris_offset = iris_position - eye_center
gaze_x = dot(iris_offset, x_axis)    (how far iris is along x axis)
gaze_y = dot(iris_offset, y_axis)    (how far iris is along y axis)
```

Now `gaze_x` and `gaze_y` are in the eye's own space — head rotation just rotates the axes along with the eye, so the measurement stays stable!

In [6]:
def get_eye_gaze(landmarks, iris_idx, inner_idx, outer_idx, top_idx, bottom_idx):
    """
    Returns (gaze_x, gaze_y) in local eye coordinate space.
    
    gaze_x: -0.5 = looking left (toward inner), +0.5 = looking right (toward outer)
    gaze_y: -0.5 = looking down, +0.5 = looking up
    
    These values are stable regardless of head position or rotation
    because they are measured relative to the eye's own geometry.
    """
    # Extract 2D positions (x, y only — z is unreliable from webcam)
    iris   = np.array([landmarks[iris_idx].x,   landmarks[iris_idx].y])
    inner  = np.array([landmarks[inner_idx].x,  landmarks[inner_idx].y])
    outer  = np.array([landmarks[outer_idx].x,  landmarks[outer_idx].y])
    top    = np.array([landmarks[top_idx].x,    landmarks[top_idx].y])
    bottom = np.array([landmarks[bottom_idx].x, landmarks[bottom_idx].y])

    # Step 1 — Eye center
    eye_center = (inner + outer) / 2.0

    # Step 2 — X axis: inner → outer, normalized
    x_axis = outer - inner
    x_norm = np.linalg.norm(x_axis)
    if x_norm < 1e-6:
        return None, None
    x_axis = x_axis / x_norm

    # Step 3 — Y axis: bottom → top, normalized
    y_axis = top - bottom
    y_norm = np.linalg.norm(y_axis)
    if y_norm < 1e-6:
        return None, None
    y_axis = y_axis / y_norm

    # Step 4 — Project iris offset onto local axes
    iris_offset = iris - eye_center
    gaze_x = float(np.dot(iris_offset, x_axis))
    gaze_y = float(np.dot(iris_offset, y_axis))

    # Normalize by eye width so distance from camera doesn't matter
    gaze_x = gaze_x / x_norm
    gaze_y = gaze_y / y_norm

    return gaze_x, gaze_y


def get_gaze_in_head(landmarks):
    """
    Returns averaged gaze vector from both eyes.
    Averaging both eyes reduces noise and handles partial occlusion.
    
    Returns (gaze_x, gaze_y) or (None, None) if landmarks unavailable.
    """
    lx, ly = get_eye_gaze(landmarks, L_IRIS, L_INNER, L_OUTER, L_TOP, L_BOTTOM)
    rx, ry = get_eye_gaze(landmarks, R_IRIS, R_INNER, R_OUTER, R_TOP, R_BOTTOM)

    # Average both eyes — if one fails, use the other
    if lx is None and rx is None:
        return None, None
    if lx is None:
        return rx, ry
    if rx is None:
        return lx, ly

    return (lx + rx) / 2.0, (ly + ry) / 2.0


print('Eye gaze functions ready!')

Eye gaze functions ready!


## Cell 5 — Smoothing

Raw gaze values are noisy — the iris detection jitters slightly frame to frame. We apply a **rolling average** over the last N frames.

**Trade-off:**
- Larger window = smoother but more lag
- Smaller window = more responsive but jittery

`SMOOTH_WINDOW = 10` is a good starting point. You can tune this later.

In [7]:
gaze_x_history = []
gaze_y_history = []


def smooth_gaze(gaze_x, gaze_y):
    """
    Applies rolling average smoothing to gaze_x and gaze_y.
    Returns smoothed (gaze_x, gaze_y).
    """
    gaze_x_history.append(gaze_x)
    gaze_y_history.append(gaze_y)

    if len(gaze_x_history) > SMOOTH_WINDOW:
        gaze_x_history.pop(0)
    if len(gaze_y_history) > SMOOTH_WINDOW:
        gaze_y_history.pop(0)

    return (
        sum(gaze_x_history) / len(gaze_x_history),
        sum(gaze_y_history) / len(gaze_y_history)
    )


def reset_smooth():
    global gaze_x_history, gaze_y_history
    gaze_x_history = []
    gaze_y_history = []


print('Smoothing ready!')

Smoothing ready!


## Cell 6 — Pinch Detection

We detect a pinch by measuring the distance between thumb tip and index tip in normalized coordinates.

**Why normalized distance?**
MediaPipe hand landmarks are already normalized 0–1 relative to frame size. So `distance < 0.05` means the tips are within 5% of the frame width — a reliable pinch threshold regardless of screen size.

**Fallback:**
If index finger goes off screen (common at frame edges), we check thumb + middle finger as fallback. This makes pinch detection robust at screen corners — exactly where you need it during calibration.

In [8]:
# Pinch state — tracks transitions to avoid repeated triggers
pinch_was_active = False

def get_landmarks(hand_landmarks):
    
    if hand_landmarks is None:
        return False, False, False

    thumb  = np.array([hand_landmarks[THUMB_TIP].x,  hand_landmarks[THUMB_TIP].y])
    index  = np.array([hand_landmarks[INDEX_TIP].x,  hand_landmarks[INDEX_TIP].y])
    middle = np.array([hand_landmarks[MIDDLE_TIP].x, hand_landmarks[MIDDLE_TIP].y])
    
    return thumb, index, middle

def get_pinch(thumb, index, middle):
    """
    Returns True if a pinch gesture is detected.
    Checks thumb+index first, falls back to thumb+middle.
    
    Uses normalized coordinates so threshold is frame-size independent.
    """
    
    if(thumb is False and index is False and middle is False):
        return False
    
    # Primary: thumb + index
    dist_index  = np.linalg.norm(thumb - index)

    # Fallback: thumb + middle (if index goes off screen)
    dist_middle = np.linalg.norm(thumb - middle)

    return dist_index < PINCH_THRESHOLD or dist_middle < PINCH_THRESHOLD


def get_pinch_event(hand_landmarks):
    """
    Returns True only on the FIRST frame of a pinch — not every frame.
    This prevents a single pinch from triggering multiple times.
    """
    global pinch_was_active
    thumb, index, middle = get_landmarks(hand_landmarks)
    is_pinching = get_pinch(thumb, middle, index)
    event       = is_pinching and not pinch_was_active
    pinch_was_active = is_pinching
    return event


print('Pinch detection ready!')

Pinch detection ready!


## Cell 7 — Calibration

### Why calibration?
Gaze-in-head gives us a stable relative measurement but it doesn't tell us where on the screen you're looking. Different people have different iris sizes, eye shapes, and sit at different distances. Calibration maps your personal gaze range to actual screen coordinates.

### How it works
We show you 5 points × 2 passes = 10 captures total:
```
Pass 1:  TL → TR → BL → BR → Center
Pass 2:  TL → TR → BL → BR → Center
```
For each point, you look at it and pinch to capture your gaze vector. After all captures, we average the two readings per point.

### The mapping
We use `np.polyfit` to fit a linear mapping from gaze space → screen space separately for X and Y:
```
screen_x = a * gaze_x + b
screen_y = c * gaze_y + d
```
This is a simple but effective approach — good enough for fundamentals and easy to understand.

In [9]:
def build_calibration_points(w, h):
    """
    Returns 5 screen points for calibration:
    4 corners + center, visited twice.
    """
    pad = 100
    points = [
        (pad,     pad),        # Top Left
        (w - pad, pad),        # Top Right
        (pad,     h - pad),    # Bottom Left
        (w - pad, h - pad),    # Bottom Right
        (w // 2,  h // 2),     # Center
    ]
    # Two passes
    return points + points


def build_calibration_map(screen_points, gaze_points):
    """
    Fits a linear mapping from gaze space to screen space.
    
    screen_points: list of (sx, sy) — where the dot was on screen
    gaze_points:   list of (gx, gy) — what gaze vector was captured
    
    Returns (x_coeffs, y_coeffs) — linear fit coefficients.
    Use np.polyval(x_coeffs, gaze_x) to get screen_x.
    """
    gx = [g[0] for g in gaze_points]
    gy = [g[1] for g in gaze_points]
    sx = [s[0] for s in screen_points]
    sy = [s[1] for s in screen_points]

    x_coeffs = np.polyfit(gx, sx, 1)   # linear: screen_x = a*gaze_x + b
    y_coeffs = np.polyfit(gy, sy, 1)   # linear: screen_y = c*gaze_y + d

    return x_coeffs, y_coeffs


def apply_calibration(gaze_x, gaze_y, x_coeffs, y_coeffs, w, h):
    """
    Maps gaze vector to screen coordinates using calibration.
    Clamps result to frame bounds.
    """
    sx = int(np.polyval(x_coeffs, gaze_x))
    sy = int(np.polyval(y_coeffs, gaze_y))
    sx = max(0, min(w, sx))
    sy = max(0, min(h, sy))
    return sx, sy


print('Calibration functions ready!')

Calibration functions ready!


## Cell 8 — Drawing Helpers

In [10]:
def draw_crosshair(frame, cx, cy, color=YELLOW, size=18):
    cv2.circle(frame, (cx, cy), size, color, 2)
    cv2.circle(frame, (cx, cy), 4,    color, -1)
    cv2.line(frame, (cx - size - 7, cy), (cx + size + 7, cy), color, 1)
    cv2.line(frame, (cx, cy - size - 7), (cx, cy + size + 7), color, 1)


def draw_calibration_target(frame, cx, cy, active=False):
    """
    Draws a calibration target dot.
    Pulses between white and cyan when waiting for pinch.
    Turns green briefly when captured.
    """
    color = GREEN if active else (CYAN if int(time.time() * 3) % 2 == 0 else WHITE)
    cv2.circle(frame, (cx, cy), 20, color, -1)
    cv2.circle(frame, (cx, cy), 24, WHITE, 2)
    cv2.circle(frame, (cx, cy), 5,  (0, 0, 0), -1)


def draw_iris_dots(frame, landmarks, w, h):
    """Draws iris center dots and eye corner dots for reference."""
    for iris_idx in [L_IRIS, R_IRIS]:
        px = int(landmarks[iris_idx].x * w)
        py = int(landmarks[iris_idx].y * h)
        cv2.circle(frame, (px, py), 4, YELLOW, -1)

    for idx in [L_INNER, L_OUTER, R_INNER, R_OUTER]:
        px = int(landmarks[idx].x * w)
        py = int(landmarks[idx].y * h)
        cv2.circle(frame, (px, py), 3, CYAN, -1)

def draw_hand_dots(frame, thumb, index, middle, w, h):
        for x, y in [thumb, index, middle]:
            cv2.circle(frame, (int(x*w), int(y*h)), 10, YELLOW, -1)
    

print('Drawing helpers ready!')

Drawing helpers ready!


## Cell 9 — Fullscreen Setup & crop_to_fill

Gets actual screen resolution and resizes frames to fill it without black bars.

In [11]:
def crop_to_fill(frame, target_w, target_h):
    src_h, src_w = frame.shape[:2]
    scale        = max(target_w / src_w, target_h / src_h)
    scaled_w     = int(src_w * scale)
    scaled_h     = int(src_h * scale)
    resized      = cv2.resize(frame, (scaled_w, scaled_h))
    x1 = (scaled_w - target_w) // 2
    y1 = (scaled_h - target_h) // 2
    return resized[y1:y1 + target_h, x1:x1 + target_w]


cap = cv2.VideoCapture(0)
ret, frame = cap.read()

cv2.namedWindow('Eye Tracking', cv2.WINDOW_NORMAL)
cv2.setWindowProperty('Eye Tracking', cv2.WND_PROP_FULLSCREEN, cv2.WINDOW_FULLSCREEN)
cv2.imshow('Eye Tracking', frame)
cv2.waitKey(1)

screen_w = cv2.getWindowImageRect('Eye Tracking')[2]
screen_h = cv2.getWindowImageRect('Eye Tracking')[3]
print(f'Screen: {screen_w}x{screen_h}')

cap.release()

Screen: 1536x864


## Cell 10 — Calibration Run

**Instructions:**
1. Look at the flashing dot
2. When your gaze is steady on it — **pinch** to capture
3. Repeat for all 10 points (5 points × 2 passes)
4. After the last point, calibration is built automatically

Press **Q** to quit early.

In [12]:
cap   = cv2.VideoCapture(0)
reset_smooth()

calib_points = build_calibration_points(screen_w, screen_h)
calib_idx    = 0
captured_screen = []   # screen positions we showed
captured_gaze   = []   # gaze vectors captured at each
flash_timer     = 0    # green flash frames after capture
calibration     = None

LABELS = ['Top Left', 'Top Right', 'Bottom Left', 'Bottom Right', 'Center']

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)
    frame = crop_to_fill(frame, screen_w, screen_h)
    h, w  = frame.shape[:2]

    timestamp = int(time.time() * 1000)
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image  = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)

    face_result = face_landmarker.detect_for_video(mp_image, timestamp)
    hand_result = hand_landmarker.detect_for_video(mp_image, timestamp)

    hand_lm = hand_result.hand_landmarks[0] if hand_result.hand_landmarks else None
    
    if(hand_lm is not None):
        thumb, index, middle = get_landmarks(hand_lm)
        draw_hand_dots(frame, thumb, middle, index, w, h)
    
    pinch   = get_pinch_event(hand_lm)

    gaze_x, gaze_y = None, None

    if face_result.face_landmarks:
        lm             = face_result.face_landmarks[0]
        raw_x, raw_y   = get_gaze_in_head(lm)

        if raw_x is not None:
            gaze_x, gaze_y = smooth_gaze(raw_x, raw_y)
            draw_iris_dots(frame, lm, w, h)

            # Show raw gaze crosshair during calibration
            raw_cx = int(w * 0.5 + gaze_x * w * 8.0)
            raw_cy = int(h * 0.5 - gaze_y * h * 8.0)
            raw_cx = max(20, min(w - 20, raw_cx))
            raw_cy = max(20, min(h - 20, raw_cy))
            draw_crosshair(frame, raw_cx, raw_cy, ORANGE)

    # Draw calibration target
    if calib_idx < len(calib_points):
        tx, ty      = calib_points[calib_idx]
        pass_num    = 1 if calib_idx < 5 else 2
        point_num   = calib_idx % 5
        label       = LABELS[point_num]

        draw_calibration_target(frame, tx, ty, active=(flash_timer > 0))

        cv2.putText(frame, f'Pass {pass_num}/2 — Look at: {label}',
                    (w//2 - 200, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.8, WHITE, 2)
        cv2.putText(frame, 'Pinch to capture',
                    (w//2 - 120, 75), cv2.FONT_HERSHEY_SIMPLEX, 0.6, YELLOW, 1)
        cv2.putText(frame, f'Point {calib_idx + 1} of {len(calib_points)}',
                    (w//2 - 80, 105), cv2.FONT_HERSHEY_SIMPLEX, 0.6, CYAN, 1)

        if flash_timer > 0:
            flash_timer -= 1

        # Capture on pinch
        if pinch and gaze_x is not None and flash_timer == 0:
            captured_screen.append((tx, ty))
            captured_gaze.append((gaze_x, gaze_y))
            flash_timer = 20
            calib_idx  += 1
            reset_smooth()

    else:
        # All points captured — build calibration
        if calibration is None:
            x_coeffs, y_coeffs = build_calibration_map(captured_screen, captured_gaze)
            calibration        = (x_coeffs, y_coeffs)
            print('Calibration complete!')
            print(f'X coeffs: {x_coeffs}')
            print(f'Y coeffs: {y_coeffs}')
            break

    cv2.imshow('Eye Tracking', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

Calibration complete!
X coeffs: [7408.40329426  479.50726238]
Y coeffs: [-4631.79142501  1281.87957657]


## Cell 11 — Live Demo with Calibrated Crosshair

Now we run the full pipeline:
1. Get face landmarks
2. Compute gaze-in-head
3. Smooth
4. Apply calibration mapping
5. Draw crosshair at mapped screen position

The crosshair should now follow your eyes accurately and stay stable when you move your head.

Press **Q** to quit.

In [13]:
if calibration is None:
    print('Run Cell 10 first to calibrate!')
else:
    x_coeffs, y_coeffs = calibration
    cap = cv2.VideoCapture(0)
    reset_smooth()

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame = cv2.flip(frame, 1)
        frame = crop_to_fill(frame, screen_w, screen_h)
        h, w  = frame.shape[:2]

        timestamp = int(time.time() * 1000)
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image  = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)

        face_result = face_landmarker.detect_for_video(mp_image, timestamp)

        if face_result.face_landmarks:
            lm           = face_result.face_landmarks[0]
            raw_x, raw_y = get_gaze_in_head(lm)

            if raw_x is not None:
                gaze_x, gaze_y = smooth_gaze(raw_x, raw_y)
                sx, sy         = apply_calibration(gaze_x, gaze_y, x_coeffs, y_coeffs, w, h)

                draw_iris_dots(frame, lm, w, h)
                draw_crosshair(frame, sx, sy, GREEN)

                # Debug values
                cv2.putText(frame, f'Gaze X: {gaze_x:.4f}', (20, h - 60),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, WHITE, 1)
                cv2.putText(frame, f'Gaze Y: {gaze_y:.4f}', (20, h - 40),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, WHITE, 1)
                cv2.putText(frame, f'Screen: ({sx}, {sy})', (20, h - 20),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, GREEN, 1)
        else:
            cv2.putText(frame, 'No face detected', (20, h - 20),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, RED, 1)

        cv2.imshow('Eye Tracking', frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()